In [ ]:
# =========================================================
# 0️⃣ INSTALL DEPENDENCIES
# =========================================================
!pip install -q albumentations==1.3.1 roboflow opencv-python seaborn thop

# =========================================================
# 1️⃣ IMPORTS & SETUP
# =========================================================
import os, glob, random, time
import numpy as np
from collections import Counter
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score
)
from sklearn.preprocessing import label_binarize

from roboflow import Roboflow
from thop import profile

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
print("🔥 Device:", device)

# =========================================================
# 2️⃣ DOWNLOAD DATASET
# =========================================================
rf = Roboflow(api_key="I2du50GUFMWkImaaH0Wm")
project = rf.workspace("ahsan-khan-iw5ip").project("blood-group-detection-kmolp")
dataset = project.version(1).download("folder")
dataset_path = dataset.location

# =========================================================
# 3️⃣ LOAD DATA & SPLIT
# =========================================================
def load_all_data(path):
    files, labels = [], []
    classes = sorted(os.listdir(os.path.join(path, "train")))
    for i, cls in enumerate(classes):
        for split in ["train", "valid", "test"]:
            p = os.path.join(path, split, cls)
            if os.path.exists(p):
                imgs = glob.glob(p + "/*")
                files.extend(imgs)
                labels.extend([i] * len(imgs))
    return files, labels, classes

files, labels, classes = load_all_data(dataset_path)

X_train, X_tmp, y_train, y_tmp = train_test_split(
    files, labels, test_size=0.30, stratify=labels, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=42
)

cnt = Counter(y_train)
print("📊 Train Distribution:", cnt)

# =========================================================
# 4️⃣ AUGMENTATION
# =========================================================
train_tfms = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(0.05, 0.1, 10, p=0.4),
    A.Normalize(),
    ToTensorV2()
])

val_tfms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(),
    ToTensorV2()
])

# =========================================================
# 5️⃣ DATASET & DATALOADERS
# =========================================================
class BloodDataset(Dataset):
    def __init__(self, files, labels, tfm):
        self.files = files
        self.labels = labels
        self.tfm = tfm

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = np.array(Image.open(self.files[idx]).convert("RGB"))
        img = self.tfm(image=img)["image"]
        return img, torch.tensor(self.labels[idx])

weights = [1.0 / cnt[l] for l in y_train]
sampler = WeightedRandomSampler(weights, len(weights), replacement=True)

train_dl = DataLoader(
    BloodDataset(X_train, y_train, train_tfms),
    batch_size=32,
    sampler=sampler,
    num_workers=2
)

val_dl = DataLoader(
    BloodDataset(X_val, y_val, val_tfms),
    batch_size=64,
    num_workers=2
)

test_dl = DataLoader(
    BloodDataset(X_test, y_test, val_tfms),
    batch_size=64,
    num_workers=2
)

# =========================================================
# 6️⃣ MODEL — MobileNetV2
# =========================================================
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, len(classes))

model = model.to(device)

print("✅ MobileNetV2 Loaded")

# =========================================================
# 7️⃣ LOSS, OPTIMIZER & SCHEDULER
# =========================================================
class_weights = torch.tensor(
    [1.0 / cnt[i] for i in range(len(classes))],
    device=device
)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25)

scaler = torch.cuda.amp.GradScaler()

# =========================================================
# 8️⃣ TRAINING
# =========================================================
EPOCHS = 25
best_val = 0

train_acc_hist, val_acc_hist = [], []
train_loss_hist, val_loss_hist = [], []
lr_hist = []

for epoch in range(EPOCHS):
    model.train()
    tp, ty = [], []
    running_loss = 0.0

    for x, y in tqdm(train_dl, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            out = model(x)
            loss = criterion(out, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        tp += out.argmax(1).cpu().tolist()
        ty += y.cpu().tolist()

    scheduler.step()
    lr_hist.append(optimizer.param_groups[0]["lr"])

    train_loss = running_loss / len(train_dl)
    tr_acc = accuracy_score(ty, tp)

    model.eval()
    vp, vy = [], []
    val_running_loss = 0.0

    with torch.no_grad():
        for x, y in val_dl:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)

            val_running_loss += loss.item()
            vp += out.argmax(1).cpu().tolist()
            vy += y.cpu().tolist()

    val_loss = val_running_loss / len(val_dl)
    v_acc = accuracy_score(vy, vp)

    train_acc_hist.append(tr_acc)
    val_acc_hist.append(v_acc)
    train_loss_hist.append(train_loss)
    val_loss_hist.append(val_loss)

    print(f"📊 Epoch {epoch+1}: Train Acc={tr_acc:.4f} | Val Acc={v_acc:.4f}")

    if v_acc > best_val:
        best_val = v_acc
        torch.save(model.state_dict(), "Best_MobileNetV2.pth")
        print("✅ Best model saved")

# =========================================================
# 9️⃣ TESTING
# =========================================================
model.load_state_dict(torch.load("Best_MobileNetV2.pth"))
model.eval()

tp, ty, y_prob = [], [], []

with torch.no_grad():
    for x, y in test_dl:
        x = x.to(device)
        out = model(x)
        prob = torch.softmax(out, dim=1)

        tp += out.argmax(1).cpu().tolist()
        ty += y.tolist()
        y_prob.extend(prob.cpu().numpy())

y_true = np.array(ty)
y_pred = np.array(tp)
y_prob = np.array(y_prob)

print("\n🎯 TEST ACCURACY:", accuracy_score(y_true, y_pred))
print("\n📄 CLASSIFICATION REPORT:\n")
print(classification_report(y_true, y_pred, target_names=classes))

# =========================================================
# 🔟 MODEL COMPLEXITY
# =========================================================
dummy = torch.randn(1,3,224,224).to(device)
flops, params = profile(model, inputs=(dummy,), verbose=False)

print("Parameters (M):", params/1e6)
print("FLOPs (GFLOPs):", flops/1e9)

# =========================================================
# 1️⃣1️⃣ INFERENCE TIME
# =========================================================
starter, ender = torch.cuda.Event(True), torch.cuda.Event(True)
starter.record()
_ = model(dummy)
ender.record()
torch.cuda.synchronize()

print("Inference Time (ms):", starter.elapsed_time(ender))

print("\n✅ MobileNetV2 Complete Pipeline Finished")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.7/125.7 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 143.7 MB/s eta 0:00:00
🔥 Device: cuda
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Blood-group-detection--1 in folder:: 100%|██████████| 14394/14394 [00:01<00:00, 7631.71it/s]


📊 Train Distribution: Counter({1: 1679, 6: 1410, 3: 1281, 5: 1266, 7: 1192, 2: 1185, 4: 1088, 0: 954})
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 73.3MB/s]


✅ MobileNetV2 Loaded


...

`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.


📊 Epoch 24: Train Acc=0.9950 | Val Acc=0.9722


Epoch 25/25:   0%|          | 0/315 [00:00<?, ?it/s]

`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.


📊 Epoch 25: Train Acc=0.9943 | Val Acc=0.9726

🎯 TEST ACCURACY: 0.9791183294663574

📄 CLASSIFICATION REPORT:

              precision    recall  f1-score   support

          A+       0.99      0.99      0.99       204
          A-       0.97      0.96      0.97       359
         AB+       0.99      0.98      0.99       254
         AB-       0.98      0.98      0.98       275
          B+       0.98      0.98      0.98       233
          B-       0.97      0.99      0.98       272
          O+       0.98      0.98      0.98       303
          O-       0.98      0.98      0.98       255

    accuracy                           0.98      2155
   macro avg       0.98      0.98      0.98      2155
weighted avg       0.98      0.98      0.98      2155

Parameters (M): 2.23412
FLOPs (GFLOPs): 0.32621696
Inference Time (ms): 9.481663703918457

✅ MobileNetV2 Complete Pipeline Finished
